In [0]:
# path = test.default.data_joins

In [0]:
dbutils.fs.ls("/Volumes/test/default/data_joins")

[FileInfo(path='dbfs:/Volumes/test/default/data_joins/new data/', name='new data/', size=0, modificationTime=1778028809873)]

In [0]:
%fs ls /Volumes/test/default/data_joins

path,name,size,modificationTime
dbfs:/Volumes/test/default/data_joins/new data/,new data/,0,1778028810440


In [0]:
base_path = "/Volumes/test/default/data_joins/new data/"

In [0]:
df_files = spark.createDataFrame(
    dbutils.fs.ls("/Volumes/test/default/data_joins/new data")
    )
display(df_files)

path,name,size,modificationTime
dbfs:/Volumes/test/default/data_joins/new data/customers_medium.csv,customers_medium.csv,38728,1778026381000
dbfs:/Volumes/test/default/data_joins/new data/menu_items.csv,menu_items.csv,6725,1778026381000
dbfs:/Volumes/test/default/data_joins/new data/order_items (2).csv,order_items (2).csv,257134,1778026381000
dbfs:/Volumes/test/default/data_joins/new data/orders_medium.csv,orders_medium.csv,286712,1778026381000
dbfs:/Volumes/test/default/data_joins/new data/restaurants.csv,restaurants.csv,3037,1778026381000


In [0]:
customers = spark.read.csv(base_path + "/customers_medium.csv", header=True, inferSchema=True)
menu = spark.read.csv(base_path + "/menu_items.csv", header=True, inferSchema=True)
order_items = spark.read.csv(base_path + "/order_items (2).csv", header=True, inferSchema=True)
orders_medium = spark.read.csv(base_path + "/orders_medium.csv", header=True, inferSchema=True)
restaurants = spark.read.csv(base_path + "/restaurants.csv", header=True, inferSchema=True)

In [0]:
order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)



In [0]:
left_df = customers.join(orders_medium,"customer_id","left")
display(left_df)

customer_id,city,signup_date,order_id,restaurant_id,order_time,delivery_time,status
C0001,Bristol,2022-04-25,O03063,R110,2023-02-09,2023-02-09T00:33:00.000Z,Delivered
C0002,London,2022-10-09,O04674,R010,2023-06-06,2023-06-06T00:33:00.000Z,Delivered
C0003,Manchester,2022-08-17,O04993,R020,2023-08-17,2023-08-17T00:56:00.000Z,Cancelled
C0004,Manchester,2022-04-15,O01521,R013,2023-03-28,2023-03-28T01:22:00.000Z,Delivered
C0005,Bristol,2023-07-13,O04533,R072,2023-11-18,2023-11-18T01:20:00.000Z,Cancelled
C0006,London,2023-08-28,O04331,R088,2023-07-22,2023-07-22T00:45:00.000Z,Delivered
C0007,Leeds,2022-02-02,O04756,R095,2024-02-29,2024-02-29T00:27:00.000Z,Delivered
C0008,London,2022-04-06,O04661,R042,2023-04-07,2023-04-07T00:32:00.000Z,Late
C0010,Liverpool,2023-09-09,O04205,R062,2023-12-15,2023-12-15T00:21:00.000Z,Cancelled
C0011,London,2023-07-29,O03554,R034,2023-03-05,2023-03-05T01:27:00.000Z,Late


In [0]:
from pyspark.sql import functions as F

In [0]:
missing_items_df = orders_medium.join (order_items,"order_id","left") \
    .filter(F.col("item_id").isNull())

In [0]:
final_df  = orders_medium \
    .join(customers,"customer_id")\
    .join(order_items,"order_id")    

display(final_df)

order_id,customer_id,restaurant_id,order_time,delivery_time,status,city,signup_date,item_id,quantity,price
O00001,C1234,R041,2023-01-17,2023-01-17T00:43:00.000Z,Late,Liverpool,2022-11-03,M0326,3,6.03
O00002,C1017,R019,2023-04-24,2023-04-24T00:33:00.000Z,Cancelled,London,2023-09-22,M0105,2,37.77
O00003,C0488,R045,2024-01-17,2024-01-17T00:29:00.000Z,Cancelled,Manchester,2022-09-28,M0243,2,25.0
O00004,C1452,R086,2023-03-27,2023-03-27T00:31:00.000Z,Late,Liverpool,2022-11-23,M0038,3,18.6
O00005,C0915,R002,2024-01-09,2024-01-09T01:02:00.000Z,Cancelled,Manchester,2022-02-10,M0344,2,24.8
O00006,C1420,R009,2023-04-18,2023-04-18T00:51:00.000Z,Cancelled,Birmingham,2022-06-08,M0037,3,29.27
O00007,C1476,R001,2023-05-12,2023-05-12T00:33:00.000Z,Delivered,Liverpool,2022-08-11,M0102,1,32.81
O00008,C0860,R022,2023-07-17,2023-07-17T01:16:00.000Z,Cancelled,Leeds,2022-10-24,M0092,2,7.08
O00009,C0720,R070,2023-09-10,2023-09-10T01:08:00.000Z,Cancelled,Manchester,2022-05-06,M0223,2,14.2
O00010,C1184,R084,2023-02-21,2023-02-21T01:21:00.000Z,Late,Manchester,2022-05-25,M0200,1,23.77


In [0]:
# Broadcast join

from pyspark.sql.functions import broadcast

optimize_df = orders_medium.join(broadcast(customers),"customer_id")

display(optimize_df)


customer_id,order_id,restaurant_id,order_time,delivery_time,status,city,signup_date
C1234,O00001,R041,2023-01-17,2023-01-17T00:43:00.000Z,Late,Liverpool,2022-11-03
C1017,O00002,R019,2023-04-24,2023-04-24T00:33:00.000Z,Cancelled,London,2023-09-22
C0488,O00003,R045,2024-01-17,2024-01-17T00:29:00.000Z,Cancelled,Manchester,2022-09-28
C1452,O00004,R086,2023-03-27,2023-03-27T00:31:00.000Z,Late,Liverpool,2022-11-23
C0915,O00005,R002,2024-01-09,2024-01-09T01:02:00.000Z,Cancelled,Manchester,2022-02-10
C1420,O00006,R009,2023-04-18,2023-04-18T00:51:00.000Z,Cancelled,Birmingham,2022-06-08
C1476,O00007,R001,2023-05-12,2023-05-12T00:33:00.000Z,Delivered,Liverpool,2022-08-11
C0860,O00008,R022,2023-07-17,2023-07-17T01:16:00.000Z,Cancelled,Leeds,2022-10-24
C0720,O00009,R070,2023-09-10,2023-09-10T01:08:00.000Z,Cancelled,Manchester,2022-05-06
C1184,O00010,R084,2023-02-21,2023-02-21T01:21:00.000Z,Late,Manchester,2022-05-25
